# Evaluasi curriculum_arm_rl di Google Colab

Notebook ini menjalankan **evaluasi saja** (tanpa training) dan menghasilkan **video MP4** sebagai pengganti MuJoCo
viewer interaktif, yang tidak bisa dibuka di Colab.

**Sebelum mulai**
1. Pilih runtime: *Runtime → Change runtime type*. **GPU (T4)** membuat rendering jauh lebih cepat (EGL). Runtime **CPU**
   juga bisa (rendering software OSMesa, lebih lambat).
2. Isi `REPO_URL` di sel parameter dengan alamat repo GitHub Anda.
3. Jalankan sel dari atas ke bawah (*Runtime → Run all* juga bisa).

Model yang dipakai: ONNX terbaru di `models/` (sudah ada di repo).

In [ ]:
!pip install onnx onnxruntime pyyaml gymnasium matplotlib

In [ ]:
# ====== PARAMETER (ubah di sini) ======
ZIP_PATH = "/content/Colab_Eval_if you want try in_Colab.zip"   # dipakai hanya bila folder proyek belum ada di /content
MISSION = 3                  # 1, 2, atau 3
MODE = "hybrid+orientasi"    # "rl" | "hybrid" (RL + LQR) | "hybrid+orientasi"; hibrida hanya untuk Misi 2 dan 3
PRECISION = False            # True -> ambang sukses T1/T2 = 5 mm (standar: 2 cm)
N_METRIK = 10                # jumlah skenario untuk tabel metrik
N_VIDEO = 2                  # jumlah skenario di dalam video
SEED0 = 1000                 # seed skenario pertama
FAST = True                  # render cepat (tanpa bayangan dan pantulan)
print("parameter tersimpan:", MISSION, MODE, PRECISION, N_METRIK, N_VIDEO, SEED0, FAST)

## 1. Pasang paket

In [ ]:
import os, subprocess, sys

if not os.environ.get("SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mujoco", "gymnasium", "stable-baselines3", "onnxruntime", "pyyaml"], check=True)
print("paket siap")

## 2. Atur rendering offscreen

MuJoCo membaca `MUJOCO_GL` saat pertama kali diimpor, jadi sel ini mengujinya di proses terpisah dan menetapkan
nilainya **sebelum** `mujoco` diimpor di notebook. Urutan: EGL (runtime GPU), lalu OSMesa (CPU).

In [ ]:
import os, shutil, subprocess, sys

_TEST = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\".1\"/></worldbody></mujoco>');"
         " d = mujoco.MjData(m); mujoco.mj_forward(m, d); r = mujoco.Renderer(m, 64, 64); r.update_scene(d); r.render(); print('GL_OK')")


def gl_ok(gl):
    r = subprocess.run([sys.executable, "-c", _TEST], env=dict(os.environ, MUJOCO_GL=gl), capture_output=True, text=True)
    return "GL_OK" in r.stdout


chosen = None
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if has_gpu:
    try:
        icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
        if not os.path.exists(icd):
            os.makedirs(os.path.dirname(icd), exist_ok=True)
            open(icd, "w").write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
    except Exception as e:
        print("ICD NVIDIA dilewati:", e)
    if gl_ok("egl"):
        chosen = "egl"
if chosen is None:
    if not gl_ok("osmesa"):
        subprocess.run(["apt-get", "install", "-y", "-qq", "libosmesa6"], capture_output=True)
    if gl_ok("osmesa"):
        chosen = "osmesa"
if chosen is None:
    raise RuntimeError("Rendering offscreen tidak tersedia (EGL dan OSMesa gagal).")
os.environ["MUJOCO_GL"] = chosen
print("GPU:", has_gpu, "| MUJOCO_GL =", chosen)

# pemeriksaan di proses ini (membuktikan urutan impor benar)
import mujoco
from mujoco import gl_context
_m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=".1"/></worldbody></mujoco>')
try:
    mujoco.Renderer(_m, 64, 64)
    _ok = "Renderer OK"
except BaseException as e:
    _ok = "Renderer GAGAL (" + type(e).__name__ + ")"
print("backend aktif:", gl_context.GLContext.__module__, "|", _ok)
if "Renderer OK" not in _ok:
    raise RuntimeError("Backend grafis salah (mujoco diimpor sebelum MUJOCO_GL diatur). "
                       "Pilih Runtime -> Restart session, lalu jalankan dari sel 0.")

## 3. Ambil proyek dari GitHub

In [ ]:
import os, sys, zipfile

ROOT_DIR = os.environ.get("COLAB_ROOT", "/content")


def find_project(root, max_depth=4):
    root = os.path.abspath(root)
    base = root.count(os.sep)
    for dp, dn, fn in os.walk(root):
        if dp.count(os.sep) - base >= max_depth:
            dn[:] = []
            continue
        dn[:] = [d for d in dn if d not in ("sample_data", "drive", "__pycache__", ".config", "hasil_colab")]
        if os.path.isfile(os.path.join(dp, "shared", "evaluation", "viewer_eval.py")):
            return dp
    return None


PROJECT_DIR = find_project(ROOT_DIR)
if PROJECT_DIR is None and os.path.isfile(ZIP_PATH):
    target = os.path.join(ROOT_DIR, "proyek")
    zipfile.ZipFile(ZIP_PATH).extractall(target)
    print("zip diekstrak ke", target)
    PROJECT_DIR = find_project(target)
if PROJECT_DIR is None:
    raise FileNotFoundError("Proyek tidak ditemukan. Unggah proyek_minimal.zip ke /content (panel Files) "
                            "atau ubah ZIP_PATH di sel 1.")
if not os.path.isfile(os.path.join(PROJECT_DIR, "shared", "evaluation", "video_render.py")):
    raise FileNotFoundError(f"{PROJECT_DIR}: shared/evaluation/video_render.py tidak ada. "
                            "Salin berkas itu dari colab_siap.zip ke folder shared/evaluation/ lalu jalankan ulang sel ini.")
os.environ["PROJECT_DIR"] = PROJECT_DIR
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("proyek:", PROJECT_DIR)
print("folder:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))
print("ONNX tersedia:", sorted(f for f in os.listdir("models") if f.endswith(".onnx")))

## 4. Metrik evaluasi (tanpa video)

Menjalankan evaluasi proyek (`shared.evaluation.run_evaluation`) secara *headless* dan mencetak ringkasan.

In [ ]:
import importlib, pathlib, yaml
from shared.evaluation import run_evaluation

EVAL_ENVS = {1: ("Mission1.eval_envs", "Stage1ReachEvalEnv"),
             2: ("Mission2.eval_envs", "Stage2PathEvalEnv"),
             3: ("Mission3.eval_envs", "Stage3ObstacleEvalEnv")}
mod, cls = EVAL_ENVS[MISSION]
EvalEnv = getattr(importlib.import_module(mod), cls)
mode = MODE if MISSION in (2, 3) else "rl"

cfg = yaml.safe_load(open(f"Mission{MISSION}/configs/eval_config.yaml", encoding="utf-8"))
cfg["viewer"].update(enabled=False, realtime_factor=0, pause_between_scenarios_sec=0, keep_open_when_done=False)
cfg["rollout"].update(n_scenarios=N_METRIK, seed_offset=SEED0)
cfg["debug"]["enabled"] = False
cfg["paths"]["log_csv"] = f"hasil_colab/eval_mission{MISSION}.csv"
if MISSION in (2, 3):
    cfg["hybrid"].update(enabled=(mode != "rl"), orientation=(mode == "hybrid+orientasi"))
if PRECISION:
    cfg["rollout"]["success_pos_threshold"] = 0.005
    if MISSION in (2, 3):
        cfg["rollout"]["t1_reach_threshold"] = 0.005
with open("/tmp/eval_colab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f)

hasil = run_evaluation(mission_id=MISSION, project_root=pathlib.Path(PROJECT_DIR).resolve(),
                       eval_cfg_path="/tmp/eval_colab.yaml", env_cls=EvalEnv)

## 5. Video && Viewer

`render_episodes` menjalankan policy dan merender tiap 2 langkah kontrol. Marker: **T1** hijau, **T2** merah, garis
T1-T2, **rintangan** oranye dengan zona aman transparan, dan jejak end-effector biru. Runtime CPU (OSMesa) lambat:
mulai dengan `N_VIDEO = 2` atau `3`.

In [ ]:
from shared.evaluation.video_render import make_policy, render_episodes, write_mp4
from shared.evaluation.viewer_eval import resolve_onnx
from shared.config_utils import load_yaml

ecfg = load_yaml(f"Mission{MISSION}/configs/eval_config.yaml")
onnx_path = resolve_onnx(pathlib.Path(".").resolve(), ecfg["paths"])
kw = {"success_pos_threshold": 0.005} if PRECISION else {}
env = EvalEnv(seed=1, **kw)
if PRECISION and MISSION in (2, 3):
    env.cfg.setdefault("path_task", {})["t1_reach_threshold"] = 0.005

policy = make_policy(onnx_path, env, mode)
frames, res = render_episodes(env, policy, list(range(SEED0, SEED0 + N_VIDEO)), fast=FAST,
                              label=f"Misi {MISSION} [{mode}]")
mp4 = write_mp4(frames, f"hasil_colab/misi{MISSION}_{mode.replace('+', '_')}.mp4", fps=25)
print(res)
print("video:", mp4, f"({os.path.getsize(mp4) // 1024} KB)")

from IPython.display import Video, display
display(Video(str(mp4), embed=True, width=720))

In [ ]:
# ====== Viewer 3D interaktif via Drake (opsional) ======
DRAKE_DETAIL = "mesh_ringan"   # "mesh_ringan" | "primitif" | "mesh_penuh"
DRAKE_MODE = "bergantian"      # "bergantian": SATU viewer, skenario diputar berurutan | "terpisah": satu viewer per skenario
N_DRAKE = 10                    # jumlah skenario
DRAKE_JEDA = 1.0               # detik jeda antar skenario (mode "bergantian")

import os, subprocess, sys, base64, uuid

# --- FUNGSI BARU: Membuat tombol untuk membuka HTML di Tab Baru ---
def button_new_tab(html_str, title="Viewer Drake"):
    """Mengonversi HTML statis menjadi Base64 dan membuat tombol klik untuk mencegah pemblokiran pop-up."""
    b64_html = base64.b64encode(html_str.encode('utf-8')).decode('utf-8')
    btn_id = "btn_" + str(uuid.uuid4()).replace("-", "")
    
    btn_html = f"""
    <a id="{btn_id}" style="display: inline-block; margin: 10px 0; padding: 10px 20px; background-color: #007bff; color: white; text-decoration: none; border-radius: 5px; font-family: sans-serif; font-weight: bold; cursor: pointer; box-shadow: 0px 2px 4px rgba(0,0,0,0.2);">
        Buka {title} di Tab Baru 🚀
    </a>
    <script>
        document.getElementById('{btn_id}').onclick = function() {{
            var base64 = '{b64_html}';
            var byteStr = atob(base64);
            var byteNumbers = new Array(byteStr.length);
            for (var i = 0; i < byteStr.length; i++) {{
                byteNumbers[i] = byteStr.charCodeAt(i);
            }}
            var byteArray = new Uint8Array(byteNumbers);
            var blob = new Blob([byteArray], {{type: 'text/html'}});
            var url = URL.createObjectURL(blob);
            window.open(url, '_blank');
        }};
    </script>
    """
    return btn_html
# ------------------------------------------------------------------

_mod = os.path.join(PROJECT_DIR, "shared", "evaluation", "drake_replay.py")
if not os.path.isfile(_mod):
    print("shared/evaluation/drake_replay.py tidak ada di proyek ini. Salin berkas itu ke folder shared/evaluation/ "
          "(lalu zip ulang atau unggah ke proyek di Colab) dan jalankan ulang sel ini. Video (sel 6) tetap berlaku.")
else:
    if not os.environ.get("SKIP_INSTALL"):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "drake"], capture_output=True, text=True)
        if r.returncode != 0:
            print("pip install drake GAGAL:\n", r.stderr[-1200:])
    from shared.evaluation import drake_replay as dr

    ok, msg = dr.drake_status()
    print(msg)
    if not ok:
        print("Viewer Drake tidak tersedia di sesi ini. Gunakan video (sel 6) sebagai gantinya.")
    else:
        from IPython.display import HTML, display
        kw_d = {"success_pos_threshold": 0.005} if PRECISION else {}
        env_d = EvalEnv(seed=1, **kw_d)
        if PRECISION and MISSION in (2, 3):
            env_d.cfg.setdefault("path_task", {})["t1_reach_threshold"] = 0.005
        policy_d = make_policy(onnx_path, env_d, mode)
        os.makedirs("hasil_colab", exist_ok=True)
        seeds_d = list(range(SEED0, SEED0 + N_DRAKE))
        rm_d = os.path.join(PROJECT_DIR, "robot_model")
        
        if DRAKE_MODE == "bergantian":
            html, info = dr.episodes_to_html(env_d, policy_d, seeds_d, rm_d, "/tmp/drake_model",
                                             detail=DRAKE_DETAIL, gap_s=DRAKE_JEDA)
            for e in info["episodes"]:
                print(e)
            print({k: v for k, v in info.items() if k != "episodes"})
            path = f"hasil_colab/replay_drake_misi{MISSION}_bergantian_seed{seeds_d[0]}-{seeds_d[-1]}.html"
            open(path, "w", encoding="utf-8").write(html)
            print("->", path)
            
            # DIGANTI: Menggunakan tombol tab baru alih-alih iframe notebook
            display(HTML(button_new_tab(html, title="Skenario Bergantian")))
        else:
            for sd in seeds_d:
                html, info = dr.episode_to_html(env_d, policy_d, sd, rm_d, "/tmp/drake_model", detail=DRAKE_DETAIL)
                path = f"hasil_colab/replay_drake_misi{MISSION}_seed{sd}.html"
                open(path, "w", encoding="utf-8").write(html)
                print(info, "->", path)
                
                # DIGANTI: Menggunakan tombol tab baru alih-alih iframe notebook
                display(HTML(button_new_tab(html, title=f"Skenario (Seed {sd})")))

## 6. Unduh hasil (opsional)

In [ ]:
try:
    from google.colab import files
    files.download(str(mp4))
    files.download(f"hasil_colab/eval_mission{MISSION}.csv")
except ImportError:
    print("Bukan di Colab; hasil ada di folder hasil_colab/")

## Pemecahan masalah

| Gejala | Penyebab / solusi |
|---|---|
| `Rendering offscreen tidak tersedia` | Pilih runtime GPU, atau jalankan ulang sel 2 (pemasangan OSMesa) |
| Video sangat lambat | Runtime CPU memakai rendering software: kurangi `N_VIDEO`, biarkan `FAST = True`, atau pakai GPU |
| `ModuleNotFoundError: shared` | Sel 3 belum dijalankan (proyek belum di-clone) |
| `Belum ada onnx_missionN_v*.onnx` | Repo tidak memuat ONNX untuk misi itu; periksa folder `models/` |
| `fatal: repository ... not found` | `REPO_URL` salah, atau repo masih Private (clone memerlukan repo Public) |
| Error `mujoco` setelah mengganti runtime | *Runtime → Restart session*, lalu jalankan dari sel 1 |